In [29]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.appName("Distributed Shared Variables")
    .master("spark://4619b568af8b:7077")
    .config("spark.pyspark.python", "/usr/bin/python3")
    .config("spark.cores.max",16)
    .config("spark.executor.cores",4)
    .config("spark.executor.memory","512M")
    .getOrCreate()
)

spark

In [30]:
# Read EMP CSV data
_schema = "first_name string, last_name string, job_title string, dob string, email string, phone string, salary double, department_id integer"
emp = spark.read.format("csv").schema(_schema).option("header", True).load("/data/csv/employee_records.csv")

## Broadcast Variables

Consider a case where we want to populate `department_name` according to the `department_id`. What options do we have?

**Option 1 — Join with a Department DataFrame:**
Create a department DataFrame and join it with the employee DataFrame to populate department names.
The problem: this invokes a **shuffle** (data exchange across executors), which is expensive.

**Option 2 — Pass a lookup variable via a UDF/map:**
Create a lookup dictionary and pass it with a UDF or map operation.
The problem: the variable gets **serialized into the task closure** and is sent **once per task** to every executor. Since there can be many tasks (one per partition), this results in redundant copies being shipped across the network, creating a **bottleneck**.

**Option 3 (Solution) — Broadcast Variable:**
When we create a broadcast variable, Spark sends it **once to each worker node** using an efficient peer-to-peer protocol (like BitTorrent). Every executor on that node then reads from the same local copy. Since each executor already has the lookup data locally, it can perform the mapping directly on its partitions — **no shuffle is needed**.

In [31]:
# Variable (Lookup)
dept_names = {1 : 'Department 1',
              2 : 'Department 2',
              3 : 'Department 3',
              4 : 'Department 4',
              5 : 'Department 5',
              6 : 'Department 6',
              7 : 'Department 7',
              8 : 'Department 8',
              9 : 'Department 9',
              10 : 'Department 10'}

In [32]:
broadcast_department_names = spark.sparkContext.broadcast(dept_names)

In [33]:
type(broadcast_department_names)

pyspark.broadcast.Broadcast

In [34]:
broadcast_department_names.value

{1: 'Department 1',
 2: 'Department 2',
 3: 'Department 3',
 4: 'Department 4',
 5: 'Department 5',
 6: 'Department 6',
 7: 'Department 7',
 8: 'Department 8',
 9: 'Department 9',
 10: 'Department 10'}

In [35]:
from pyspark.sql.functions import udf

@udf
def get_dept_names(department_id):
    return broadcast_department_names.value.get(department_id)

In [36]:
from pyspark.sql.functions import col
emp_final = emp.withColumn("dept_name",get_dept_names(col("department_id")))

In [37]:
emp_final.show()

+----------+----------+--------------------+----------+--------------------+--------------------+--------+-------------+-------------+
|first_name| last_name|           job_title|       dob|               email|               phone|  salary|department_id|    dept_name|
+----------+----------+--------------------+----------+--------------------+--------------------+--------+-------------+-------------+
|   Richard|  Morrison|Public relations ...|1973-05-05|melissagarcia@exa...|       (699)525-4827|512653.0|            8| Department 8|
|     Bobby|  Mccarthy|   Barrister's clerk|1974-04-25|   llara@example.net|  (750)846-1602x7458|999836.0|            7| Department 7|
|    Dennis|    Norman|Land/geomatics su...|1990-06-24| jturner@example.net|    873.820.0518x825|131900.0|           10|Department 10|
|      John|    Monroe|        Retail buyer|1968-06-16|  erik33@example.net|    820-813-0557x624|485506.0|            1| Department 1|
|  Michelle|   Elliott|      Air cabin crew|1975-03-31|

In [ ]:
# Accumulators are another kind of distributed shared variable

In [38]:
# spark.stop()